In [ ]:
import sys
sys.path.append("..")
import torch

from src.environment.proto import MultiCurrencyEnv
from src.data import load_data, load_and_align_data, PAIRS, get_field

from bokeh.palettes import Category10
import bokeh.plotting as bk
bk.output_notebook()

# Read Historical Data

In [ ]:
PAIRS = {
    'Bitcoin': 'XBTEUR',
    'Ethereum': 'ETHEUR',
    'Ripple': 'XRPEUR',
    'Cardano': 'ADAEUR'
}

In [ ]:
data, times = load_and_align_data(PAIRS, interval=30)

In [ ]:
times_ = torch.tensor([t.timestamp() for t in times])
dt = float(times_.diff().mean())
print(f"dt = {dt}")
prices = torch.tensor(get_field(data, 'open')).T
volume = torch.tensor(get_field(data, 'volume')).T

history = [{
    'time'  : t,
    'prices': p,
    'volume': v
} for t, p, v in zip(times_, prices, volume)]

history = sorted(history, key=lambda x: x['time'])

len(history)

In [ ]:
f1 = bk.figure(title=f"Prices", x_axis_type="datetime", x_axis_label="t", y_axis_label="USD", width=1200, height=500, tools="pan,wheel_zoom,box_zoom,reset,save,hover")

for i, (name, price) in enumerate(zip(data.keys(), prices.T)):
    r = f1.line(times[::1000], price[::1000], line_width=2, legend_label=f"{name}", color=Category10[10][i%10])
f1.legend.click_policy = "hide"

bk.show(f1)

In [ ]:
tau_p = torch.tensor([dt*10, dt*60*6, dt*60*24*3, dt*60*24*30])
tau_s = torch.tensor([dt*10, dt*60*6])
tau_v = torch.tensor([dt*10, dt*60*6])

env = MultiCurrencyEnv(
    N=len(data),
    C0=1_000.0,
    tau_p=tau_p,
    tau_s=tau_s,
    tau_v=tau_v,
    sell_fee=0.005,
    buy_fee=0.005,
    reward_mode="log",
    transaction_eps=1e-3,
    save_history=True
)

state = env.reset(history[0])
for elem in history[1:1000]:
    a = torch.tanh(torch.randn(env.action_size))

    state, reward, done, info = env.step(a, data=elem)
    if done:
        break

# Integrate Agent

In [ ]:
env.state_size, env.action_size

In [ ]:
from src.network import VanillaNetwork, VanillaQNetwork
from src.agent.ddpg import DDPGConfig, VanillaDDPG


config = DDPGConfig(
    gamma=0.99,
    tau=100.0,
    noise_std=0.1,
    buffer_size=1000000
)

agent = VanillaDDPG(
    actor=VanillaNetwork(env.state_size, env.action_size, sizes=[64, 64], act=torch.tanh, out_act=torch.tanh),
    critic=VanillaQNetwork(env.state_size, env.action_size, sizes=[64, 64], act=torch.tanh),
    config=config
)

In [ ]:
# agent.load("../data/agent/ddpg1.ptm")

In [ ]:
def train_on_historical(
        agent, env, data,
        n_episodes,
        batch_size=32,
        update_interval=100,
        n_updates=8,
        max_steps=2000,
        warm_up=500,
        store=1,
        actor_lr=1e-4,
        critic_lr=1e-4,
        actor_m=0.0,
        critic_m=0.0):
    """Train the agent in the given environment."""
    agent.update_optimizers(actor_lr, critic_lr, actor_m, critic_m)

    if store:
        total_reward = []
        total_info = []
        total_loss = []

    for episode in range(1, n_episodes+1):
        start = torch.randint(len(data)-max_steps, size=[1]).item()
        state = env.reset(history[start]).to_tensor()

        if store:
            episode_reward = []
            episode_info = []
            episode_loss = []

        for i in range(max_steps):
            if i < warm_up:
                action = torch.zeros(env.action_size)
            else:
                action = agent.act(state, explore=True)

            next_state, reward, done, info = env.step(action, history[start+i])
            next_state = next_state.to_tensor()

            episode_reward.append(reward)
            episode_info.append(info)

            if i > warm_up:
                agent.buffer.store(
                    state.detach(),
                    action.detach(),
                    next_state.detach(),
                    torch.tensor(reward).to(env.dtype),
                    torch.tensor(done).to(env.dtype)
                )
        
                if (i+1) % update_interval == 0:
                    loss_dicts = []
                    for _ in range(n_updates):
                        loss_dicts.append(agent.update(batch_size))
                    
                    loss_dict = {
                        key: sum([item[key] for item in loss_dicts])
                    for key in loss_dicts[0]}

                    msg = f"episode {episode} - reward: {sum(episode_reward):.2f}"
                    for key, val in loss_dict.items():
                        msg += f" - {key}: {val:.5f}"
                    print(msg, end="\r")

            if done or (i+1) > max_steps:
                break

            state = next_state

        if store:
            total_loss.append(episode_loss)
            total_reward.append(episode_reward)
            total_info.append(episode_info)

        print(f"episode {episode} - total reward: {sum(episode_reward):.5f}" + " "*100)
    
    if store:
        return (
            total_loss,
            total_reward,
            total_info,
        )
    
    return [], [], [], {}

In [ ]:
loss, rewards, info = train_on_historical(agent, env, history, n_episodes=10, batch_size=32, n_updates=16, max_steps=20000, warm_up=1000)

In [ ]:
agent.save("../data/agent/ddpg1.ptm")

In [ ]:
loss_a = torch.tensor([[loss_dict['actor_loss'] for loss_dict in episode_loss] for episode_loss in loss])

fig = bk.figure(title="Actor Losses", x_axis_label="Training Iteration [Epochs]", y_axis_label="Loss", width=900, height=320)
fig.line(torch.arange(loss_a.numel()) / loss_a.shape[1], loss_a.flatten(), line_width=2, legend_label="Loss / Batch", color=Category10[10][0], alpha=0.3)
fig.line(torch.arange(len(loss_a)) + 0.5, loss_a.mean(1), line_width=2, legend_label="Epoch Average", color=Category10[10][0])
bk.show(fig)

In [ ]:
loss_c = torch.tensor([[loss_dict['critic_loss'] for loss_dict in episode_loss] for episode_loss in loss])

fig = bk.figure(title="Critic Losses", x_axis_label="Training Iteration [Epochs]", y_axis_label="Loss", width=900, height=320, tools="pan,wheel_zoom,box_zoom,reset,save,hover")
fig.line(torch.arange(loss_c.numel()) / loss_c.shape[1], loss_c.flatten(), line_width=2, legend_label="Loss / Batch", color=Category10[10][2], alpha=0.3)
fig.line(torch.arange(len(loss_c)) + 0.5, loss_c.mean(1), line_width=2, legend_label="Epoch Average", color=Category10[10][2])
bk.show(fig)

In [ ]:
rewards = torch.tensor(rewards)

fig = bk.figure(title="Rewards", x_axis_label="Training Iteration [Episodes]", y_axis_label="Loss", width=900, height=320)
fig.line(torch.arange(rewards.numel()) / rewards.shape[1], rewards.flatten(), line_width=2, legend_label="Reward", color=Category10[10][4], alpha=0.3)
fig.line(torch.arange(len(rewards)) + 0.5, rewards.mean(1), line_width=2, legend_label="Average Reward / Episode", color=Category10[10][4])
bk.show(fig)

In [ ]:
episode = 7

Vs = [item['V'] for item in info[episode]]
Cs = [item['C'] for item in info[episode]]
ts = [i for i in range(len(Vs))]

f1 = bk.figure(title=f"Episode {episode} - Portfolio Value", x_axis_label="t", y_axis_label="USD", width=900, height=320)
r1 = f1.line(ts, Vs, line_width=2, legend_label="Total V")
r2 = f1.line(ts, Cs, line_width=1, line_dash="dashed", legend_label="Cash C")
f1.legend.click_policy = "hide"
bk.show(f1)

fig = bk.figure(title=f"Episode {episode} - Rewards", x_axis_label="t", y_axis_label="Reward (log(V_t+1/V_t))", width=900, height=320)
fig.line(ts, rewards[episode], line_width=2, color=Category10[10][4])
bk.show(fig)

hist, edges = torch.histogram(rewards[episode], bins=200, density=True)
x = (edges[:-1] + edges[1:]) / 2; y = hist
p = bk.figure(title="Histogram as Line with Shaded Fill", width=900, height=320)
p.varea(x=x, y1=0, y2=y, fill_color=Category10[10][4], fill_alpha=0.4)
p.line(x, y, line_color=Category10[10][4], line_width=2)
bk.show(p)

In [ ]:
actions = torch.tensor([item['a'] for item in info[episode]])
ts = [i for i in range(len(actions))]
skip = 10

f1 = bk.figure(title=f"Action Fraction", x_axis_label="t", y_axis_label="USD", width=900, height=320)
f1.scatter(ts[::skip], actions[::skip,0], size=1, legend_label="a_0")
# for i in range(1,env.N+1):
#     f1.scatter(ts[::skip], actions[::skip,i], size=1, legend_label=f"a_{i}", color=Category10[10][(i+1)%10])
f1.legend.click_policy = "hide"
bk.show(f1)